## **cleaning**

In [ ]:
# Installations
!pip install emoji_data_python

import re
import emoji_data_python as edp

In [ ]:
input_path = "/content/CM_train_nolabel.tsv"
output_path = "/content/CM_cleaned_01.txt"

with open(input_path, "r", encoding="utf-8", errors="ignore") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:
    for i, line in enumerate(fin):
        if i >= 5100:
            break
        fout.write(re.sub(r" +", " ", line.strip().lower()) + "\n")

# read 5100 lines
# lower cased english characters
# removed extra spaces between and ends

In [ ]:
input_path = "/content/CM_cleaned_01.txt"
output_path = "/content/CM_cleaned_02.txt"

emoji_char_map = {e.char: e.short_name for e in edp.emoji_data}

with open(input_path, "r", encoding="utf-8", errors="ignore") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for row_no, line in enumerate(fin, start=1):
        s = line.rstrip("\n")
        out = []
        i = 0
        n = len(s)

        while i < n:
            ch = s[i]

            if ch in emoji_char_map:
                j = i + 1
                while j < n and s[j] == ch:
                    j += 1
                short = emoji_char_map[ch]
                word = EMOJI_REDUCE_MAP.get(short, short)
                out.append(" ")
                out.append(word)
                out.append(" ")
                i = j
            else:
                out.append(ch)
                i += 1

        cleaned = re.sub(r" +", " ", "".join(out)).strip()
        fout.write(cleaned + "\n")

# emoji replaced by simple single word

In [ ]:
input_path = "/content/CM_cleaned_02.txt"
output_path = "/content/CM_cleaned_03.txt"

allowed_pattern = re.compile(r"[A-Za-z0-9\u0D00-\u0D7F ]")
remove_pattern = re.compile(r"[^A-Za-z0-9\u0D00-\u0D7F ]")

with open(input_path, "r", encoding="utf-8", errors="ignore") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for row_no, line in enumerate(fin, start=1):
        s = line.rstrip("\n")

        # removed = "".join(remove_pattern.findall(s))
        # if removed:
        #     print(f"{row_no} | {removed}")

        cleaned = remove_pattern.sub(" ", s)
        cleaned = re.sub(r" +", " ", cleaned).strip()
        if (len(cleaned) <= 10) :
            fout.write("THIS SENTENCE HAS BEEN REMOVED" + "\n")
            # print(row_no, " ", cleaned)
            continue;

        fout.write(cleaned + "\n")

# removed non-alpha numeric, non-malayalam (tamil, arabic, kannada, telugu )
# removed smaller sentences (THIS SENTENCE HAS BEEN REMOVED)

## **Amazon CM**

In [ ]:
import pandas as pd

df = pd.read_csv('/content/amazon_cm_01.csv', nrows=10000)
df.to_csv('/content/amazon_cm_02.csv', index=False)


## **Google Translate API**

In [ ]:
!pip install google-cloud-translate
!pip install indic-transliteration

In [ ]:
import os
from google.cloud import translate_v2 as translate

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my-key.json"

def translate_sentences(sentences):
    client = translate.Client()

    translations = []

    for sentence in sentences:
        result = client.translate(
            sentence,
            source_language="ml",
            target_language="en",
            format_="text"
        )
        translations.append(result["translatedText"])

    return translations


manglish_sentences = list(x for x in open("/content/test.csv", "r", errors="ignore"))[0:10];

translated = translate_sentences(manglish_sentences)

for i, t in enumerate(translated):
    print(f"{i+1}. {t}")
# Using Google API

## **IndicTrans2**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/CM_10K_translated.csv")
df = df.iloc[:, :-1]

df.to_csv("/content/CM_10K.csv", index=False)

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers==4.40.2" accelerate sentencepiece onnx onnxruntime
!pip install -q IndicTransToolkit

In [ ]:
import os
os.environ["HF_TOKEN"] = ""

In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

INPUT_PATH = "/content/CM_10K.csv"
OUTPUT_PATH = "/content/CM_10K_Indic.csv"
BATCH_SIZE = 5

src_lang = "mal_Mlym"
tgt_lang = "eng_Latn"
model_name = "ai4bharat/indictrans2-indic-en-1B"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv(INPUT_PATH)

# If output file exists, resume from it
if os.path.exists(OUTPUT_PATH):
    print("Resuming from existing file...")
    df_out = pd.read_csv(OUTPUT_PATH)
else:
    print("Starting fresh...")
    df_out = df.copy()
    df_out["translated_text"] = None

total_rows = len(df_out)

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2"
).to(DEVICE)

ip = IndicProcessor(inference=True)

start_index = df_out["translated_text"].first_valid_index()

if start_index is None:
    start_index = 0
else:
    # Continue after last filled row
    start_index = df_out["translated_text"].last_valid_index() + 1

print(f"Starting from row: {start_index}")


for i in range(start_index, total_rows, BATCH_SIZE):

    try:
        print(f"\nProcessing rows {i} to {min(i+BATCH_SIZE, total_rows)}")

        batch_sentences = (
            df_out.iloc[i:i+BATCH_SIZE, 1]
            .astype(str)
            .tolist()
        )

        # Preprocess
        batch = ip.preprocess_batch(
            batch_sentences,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )

        # Tokenize
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode
        decoded = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess
        translations = ip.postprocess_batch(
            decoded,
            lang=tgt_lang
        )

        # Save batch output
        df_out.loc[i:i+len(translations)-1, "translated_text"] = translations

        # Save to CSV after every batch
        df_out.to_csv(OUTPUT_PATH, index=False)

        print("Batch saved successfully.")

    except Exception as e:
        print(f"Error occurred at batch starting row {i}")
        print(str(e))
        print("You can re-run the script. It will resume automatically.")
        break

print("\nTranslation process completed (or stopped safely).")


## **Cleaning + Transliterate + Translate with Google API**

In [ ]:
!pip install -q indic-transliteration emoji_data_python google-cloud-translate==2.0.1

In [ ]:
import re

def basic_clean(input_path, output_path):
    with open(input_path, "r", encoding="utf-8", errors="ignore") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for i, line in enumerate(fin):
            cleaned = re.sub(r" +", " ", line.strip().lower())
            fout.write(cleaned + "\n")

# Run
basic_clean("/content/CM_train_nolabel.tsv", "/content/step1.txt")


In [ ]:
import re
import emoji_data_python as edp

def remove_emojis(input_path, output_path):
    emoji_set = set(e.char for e in edp.emoji_data)

    with open(input_path, "r", encoding="utf-8", errors="ignore") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            cleaned = "".join(ch for ch in line if ch not in emoji_set)
            cleaned = re.sub(r" +", " ", cleaned).strip()
            fout.write(cleaned + "\n")

# Run
remove_emojis("/content/step1.txt", "/content/step2.txt")


In [ ]:
import re

def remove_special_chars(input_path, output_path):
    remove_pattern = re.compile(r"[^A-Za-z0-9\u0D00-\u0D7F ]")

    with open(input_path, "r", encoding="utf-8", errors="ignore") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            cleaned = remove_pattern.sub(" ", line.strip())
            cleaned = re.sub(r" +", " ", cleaned).strip()

            if len(cleaned) <= 20:
                continue

            fout.write(cleaned + "\n")

# Run
remove_special_chars("/content/step2.txt", "/content/step3.txt")


In [ ]:
import pandas as pd
def txt_to_csv(txt_path, csv_path, column_name="text"):
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    df = pd.DataFrame({column_name: lines})
    df.to_csv(csv_path, index=False)

    # Run
txt_to_csv("/content/step3.txt", "/content/processed_data.csv")

In [ ]:
import pandas as pd
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def add_transliteration_column(csv_path, text_column="text"):
    df = pd.read_csv(csv_path)

    def translit_safe(x):
        try:
            return transliterate(
                str(x),
                sanscript.ITRANS,
                sanscript.MALAYALAM
            )
        except:
            return ""

    df["transliterated"] = df[text_column].apply(translit_safe)

    df.to_csv(csv_path, index=False)

# Run
add_transliteration_column("/content/processed_data.csv", text_column="text")


In [ ]:
import os
import pandas as pd
from google.cloud import translate_v2 as translate

def batch_translate_and_save(csv_path,
                             credential_path,
                             text_column="transliterated",
                             output_column="translated_en",
                             total_limit=10000,
                             batch_size=100):

    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credential_path
    client = translate.Client()

    df = pd.read_csv(csv_path)

    if output_column not in df.columns:
        df[output_column] = ""

    rows_to_process = min(total_limit, len(df))

    for start in range(0, rows_to_process, batch_size):
        end = min(start + batch_size, rows_to_process)

        batch_texts = df[text_column].iloc[start:end].tolist()

        results = client.translate(
            batch_texts,
            source_language="ml",
            target_language="en",
            format_="text"
        )

        for i, result in enumerate(results):
            df.loc[start + i, output_column] = result["translatedText"]

        df.to_csv(csv_path, index=False)

        print(f"Saved batch {start} to {end}")

# Run
batch_translate_and_save(
    "/content/processed_data.csv",
    "my-key.json"
)


## **transliteration**

In [ ]:
import pandas as pd
import time
import os
from gradio_client import Client
from google.cloud import translate_v2 as translate

INPUT_FILE = "CM_10K_subset_selected.csv"
OUTPUT_FILE = "CM_10K_translated_updated.csv"
BATCH_SIZE = 20

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my-key.json"

if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE)

if "transliterated_ml" not in df.columns:
    df["transliterated_ml"] = ""

if "translated_en" not in df.columns:
    df["translated_en"] = ""

source_sentences = df["text"].astype(str).tolist()

translit_client = Client("vrclc/en-ml-transliteration")
translate_client = translate.Client()

start_index_translit = 2200
print(start_index_translit)
if pd.isna(start_index_translit):
    start_index_translit = len(df)

for i in range(int(start_index_translit), len(df), BATCH_SIZE):
    batch_indices = range(i, min(i + BATCH_SIZE, len(df)))
    for idx in batch_indices:
        # if df.at[idx, "transliterated_ml"] == "":
            try:
                result = translit_client.predict(source_sentences[idx])
                df.at[idx, "transliterated_ml"] = result
                print(idx)
                print(source_sentences[idx])
                print(result)
            except Exception as e:
                print("Error in transliteration:", e)
    df.to_csv(OUTPUT_FILE, index=False)
    time.sleep(1)

start_index_translate = df[df["translated_en"] == ""].index.min()
if pd.isna(start_index_translate):
    start_index_translate = len(df)

for i in range(int(start_index_translate), len(df), BATCH_SIZE):
    batch_indices = range(i, min(i + BATCH_SIZE, len(df)))
    for idx in batch_indices:
        if df.at[idx, "translated_en"] == "":
            try:
                result = translate_client.translate(
                    df.at[idx, "transliterated_ml"],
                    source_language="ml",
                    target_language="en",
                    format_="text"
                )
                translated_text = result["translatedText"]
                df.at[idx, "translated_en"] = translated_text
                print("Translated:", translated_text)
            except Exception as e:
                print("Error in translation:", e)
    df.to_csv(OUTPUT_FILE, index=False)
    time.sleep(1)

print("Completed and saved to", OUTPUT_FILE)

In [ ]:
import pandas as pd
import time
import os
from google.cloud import translate_v2 as translate

INPUT_FILE = "CM_10K_translated_updated.csv"
OUTPUT_FILE = "CM_10K_translated_updated.csv"
BATCH_SIZE = 20

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my-key.json"

df = pd.read_csv(INPUT_FILE)

if "translated_ml" not in df.columns:
    df["translated_ml"] = ""

translate_client = translate.Client()

start_index_translate = df[df["translated_ml"] == ""].index.min()
if pd.isna(start_index_translate):
    start_index_translate = len(df)

for i in range(int(start_index_translate), len(df), BATCH_SIZE):
    batch_indices = range(i, min(i + BATCH_SIZE, len(df)))
    for idx in batch_indices:
        if df.at[idx, "translated_ml"] == "":
            try:
                result = translate_client.translate(
                    df.at[idx, "transliterated_ml"],
                    source_language="ml",
                    target_language="en",
                    format_="text"
                )
                translated_text = result["translatedText"]
                df.at[idx, "translated_ml"] = translated_text
                print("Translated:", translated_text)
            except Exception as e:
                print("Error in translation:", e)
    df.to_csv(OUTPUT_FILE, index=False)
    time.sleep(1)

print("Translation completed and saved.")

## **Code Mix Ratio**

In [ ]:
!pip install wordfreq

In [ ]:
import pandas as pd
import re
from wordfreq import zipf_frequency

def detect_lang(token):
    token = token.strip()

    # Malayalam script characters
    if re.search(r"[\u0D00-\u0D7F]", token):
        return "ml"

    # Pure alphabetic (Roman script)
    if re.match("^[A-Za-z]+$", token):
        if zipf_frequency(token.lower(), 'en') > 3:
            return "en"
        else:
            return "ml"
    return "other"

def tokenize(sentence):
    return str(sentence).split()

def get_tags(sentence):
    tokens = tokenize(sentence)
    tags = [detect_lang(t) for t in tokens]
    return tokens, tags

def compute_metrics(tags):
    filtered = [t for t in tags if t != "other"]
    if len(filtered) == 0:
        return 0, 0, 0

    total = len(filtered)
    en_count = filtered.count("en")
    ml_count = filtered.count("ml")

    en_ratio = en_count / total
    dominant = max(en_count, ml_count)
    cmi = (total - dominant) / total

    switches = sum(
        1 for i in range(1, len(filtered))
        if filtered[i] != filtered[i-1]
    )
    switch_rate = switches / total

    return en_ratio, cmi, switch_rate

MALAYALAM_PATTERN = r"[\u0D00-\u0D7F]"
ENGLISH_PATTERN = r"[A-Za-z]"

def detect_sentence_script(sentence):
    sentence = str(sentence)
    has_mal = bool(re.search(MALAYALAM_PATTERN, sentence))
    has_eng = bool(re.search(ENGLISH_PATTERN, sentence))
    if has_mal and not has_eng:
        return "mal"
    elif has_eng and not has_mal:
        return "eng"
    else:
        return "mix"

def cm_category(ratio):
    if ratio <= 0.2:
        return "lowcm"
    elif ratio <= 0.5:
        return "midcm"
    else:
        return "highcm"

def cs_category(ratio):
    if ratio <= 0.3:
        return "lowcs"
    else:
        return "highcs"

df = pd.read_csv("/content/processed_data.csv")

results = []

for text in df["text"]:
    tokens, tags = get_tags(text)
    en_ratio, cmi, switch_rate = compute_metrics(tags)

    results.append({
        "text": text,
        "tokens": tokens,
        "tags": tags,
        "CMI": cmi,
        "switch_rate": switch_rate,
        "script_type": detect_sentence_script(text),
        "cm_category": cm_category(cmi),
        "cs_category": cs_category(switch_rate),
    })

final_df = pd.DataFrame(results)

final_df["strata"] = (
    final_df["script_type"] + "_" +
    final_df["cm_category"] + "_" +
    final_df["cs_category"]
)

target_size = 10000

strata_counts = final_df["strata"].value_counts()
strata_proportions = strata_counts / len(final_df)
strata_target_counts = (strata_proportions * target_size).round().astype(int)

subset_list = []

for stratum, n_samples in strata_target_counts.items():
    stratum_df = final_df[final_df["strata"] == stratum]
    n_samples = min(n_samples, len(stratum_df))
    subset_list.append(
        stratum_df.sample(n=n_samples, random_state=42)
    )

subset = pd.concat(subset_list).reset_index(drop=True)

subset.to_csv("CM_10K_subset_selected.csv")

In [ ]:
# verification
orig_dist = final_df["strata"].value_counts(normalize=True)
subset_dist = subset["strata"].value_counts(normalize=True)

comparison = pd.DataFrame({
    "original": orig_dist,
    "subset": subset_dist
}).fillna(0)

comparison

## **Model**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/CM_10K_subsetselected_translated_updated.csv")

# remove empty translations
df = df[df["translated_ml"].notna()]
df = df[df["translated_ml"] != ""]

df = df[["text","translated_ml"]]
df = df.rename(columns={
    "text":"source",
    "translated_ml":"target"
})

df.to_csv("train.csv",index=False)

In [ ]:
!pip install evaluate

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

df = pd.read_csv("train-01.csv")[0:4500]

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

model_name = "google/byt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# store fewer activations, & recomputes during backward pass, saves gpu memory
model.gradient_checkpointing_enable()
model.config.use_cache = False #cache breaks gradient checkpointing

max_len = 64

def preprocess(example): #->{input_ids, attention_mask, labels}

    model_inputs = tokenizer(
        example["source"],
        truncation=True,
        max_length=max_len
    )

    labels = tokenizer(
        example["target"],
        truncation=True,
        max_length=max_len
    )["input_ids"]

    # for labels -100 pads, to ignore in loss calculation
    labels = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels
    ]

    model_inputs["labels"] = labels

    return model_inputs


train_dataset = train_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])

# pads sentences
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

bleu = evaluate.load("bleu")

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    labels = [[l for l in label if l != -100] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    filtered_preds = []
    filtered_labels = []

    for p, l in zip(decoded_preds, decoded_labels):
        if l.strip() != "":
            filtered_preds.append(p)
            filtered_labels.append([l])

    if len(filtered_preds) == 0:
        return {"bleu": 0.0}

    result = bleu.compute(
        predictions=filtered_preds,
        references=filtered_labels
    )

    return {"bleu": result["bleu"]}


training_args = Seq2SeqTrainingArguments(
    output_dir="./cm_byt5_model",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    remove_unused_columns=False,
    predict_with_generate=True,
    generation_max_length=max_len,
    eval_strategy="epoch"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
trainer.evaluate()

trainer.save_model("./cm_byt5_model")
tokenizer.save_pretrained("./cm_byt5_model")